In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
client = bigquery.Client(project="sous-nos-pieds")

In [ ]:

# Projection des coûts futurs de sinistres (Inondation, Sécheresse, Mvt Terrain)...........................................

import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project="sous-nos-pieds")


# 1. Fréquence annuelle des arrêtés par aléa..............................................................

query_catnat = """
SELECT
  lib_risque_jo,
  EXTRACT(YEAR FROM date_debut) AS annee
FROM `sous-nos-pieds.staging.catnat_staging`
WHERE date_debut IS NOT NULL
"""
df_catnat = client.query(query_catnat).to_dataframe()
mapping = {
    "inondations et/ou coulees de boue": "Inondation",
    "inondations remontee nappe": "Inondation",
    "coulee de boue": "Inondation",
    "secheresse": "Secheresse",
    "mouvement de terrain": "MvtTerrain",
    "glissement de terrain": "MvtTerrain",
    "effondrement et/ou affaisement": "MvtTerrain",
    "eboulement et/ou chute de blocs": "MvtTerrain",
    "lave torrentielle": "MvtTerrain",
    "glissement et effondrement de terrain": "MvtTerrain",
    "glissement et eboulement rocheux": "MvtTerrain",
}
df_catnat["cat"] = df_catnat["lib_risque_jo"].map(mapping)
df_catnat["cat"] = df_catnat["lib_risque_jo"].map(mapping)
df_3aleas = df_catnat[df_catnat["cat"].notna()]
freq_annuelle = df_3aleas.groupby(["cat", "annee"]).size().unstack(0, fill_value=0)

# Fenêtre récente stable (exclut la queue 2023-2025, incomplète par délai de reconnaissance administrative)...............................
freq_recente = freq_annuelle.loc[2013:2022].mean()

# 2. Coût moyen historique par arrêté (1995-2021)..............................................................

# Remplace "financier_marts.TABLE_NOM" par le nom exact de la table de ton équipe......................................
query_cost = """
SELECT
  cout_cumule_inondation_1995_2021,
  cout_cumule_secheresse_rga_1995_2021,
  cout_cumule_mvt_terrain_1995_2021,
  departement_code,
  departement_nom
FROM `sous-nos-pieds.marts.financier_marts`
"""
df_cost = client.query(query_cost).to_dataframe()
df_cost = client.query(query_cost).to_dataframe()

milieux = {
    "Pas de sinistre répertorié à CCR": 0, "Entre 0 k€ et 100 k€": 50_000,
    "Entre 100 k€ et 500 k€": 300_000, "Entre 500 k€ et 2 M€": 1_250_000,
    "Entre 2 M€ et 5 M€": 3_500_000, "Entre 5 M€ et 10 M€": 7_500_000,
    "Entre 10 M€ et 50 M€": 30_000_000, "Entre 50 M€ et 100 M€": 75_000_000,
    "Supérieur à 100 M€": 150_000_000,
}
cout_total = {
    "Inondation": df_cost["cout_cumule_inondation_1995_2021"].map(milieux).sum(),
    "Secheresse": df_cost["cout_cumule_secheresse_rga_1995_2021"].map(milieux).sum(),
    "MvtTerrain": df_cost["cout_cumule_mvt_terrain_1995_2021"].map(milieux).sum(),
}
nb_arretes_periode = df_3aleas[df_3aleas["annee"].between(1995, 2021)].groupby("cat").size()

cout_moyen_par_arrete = {cat: cout_total[cat] / nb_arretes_periode[cat] for cat in cout_total}


# 3. Projection nationale (par année, 5 ans, 10 ans)..........................................................................

projection_nationale = []
for cat in ["Inondation", "Secheresse", "MvtTerrain"]:
    cout_annuel = freq_recente[cat] * cout_moyen_par_arrete[cat]
    projection_nationale.append({
        "alea": cat,
        "frequence_annuelle_projetee": round(freq_recente[cat]),
        "cout_moyen_par_arrete_eur": round(cout_moyen_par_arrete[cat]),
        "cout_projete_annuel_eur": round(cout_annuel),
        "cout_projete_5ans_eur": round(cout_annuel * 5),
        "cout_projete_10ans_eur": round(cout_annuel * 10),
    })
df_projection = pd.DataFrame(projection_nationale)
print(df_projection)


# 4. Projection par département (répartition selon le poids historique de chaque département)................................

df_cost["cout_inondation_est"] = df_cost["cout_cumule_inondation_1995_2021"].map(milieux)
df_cost["cout_secheresse_est"] = df_cost["cout_cumule_secheresse_rga_1995_2021"].map(milieux)
df_cost["cout_mvt_est"] = df_cost["cout_cumule_mvt_terrain_1995_2021"].map(milieux)

poids_dept = df_cost.groupby(["departement_code", "departement_nom"]).agg(
    cout_inondation_est=("cout_inondation_est", "sum"),
    cout_secheresse_est=("cout_secheresse_est", "sum"),
    cout_mvt_est=("cout_mvt_est", "sum"),
).reset_index()

for cat, col in [("Inondation", "cout_inondation_est"), ("Secheresse", "cout_secheresse_est"), ("MvtTerrain", "cout_mvt_est")]:
    part = poids_dept[col] / poids_dept[col].sum()
    cout_annuel_national = freq_recente[cat] * cout_moyen_par_arrete[cat]
    poids_dept[f"projection_{cat.lower()}_annuelle_eur"] = round(part * cout_annuel_national)

print(poids_dept.head(10))

# 5. Export vers BigQuery pour Looker............................................................

job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")

client.load_table_from_dataframe(
    df_projection,
    "wagon-bootcamp-501621.risques_naturels.projection_couts_futurs",
    job_config=job_config
).result()
print("✓ projection_couts_futurs exportée")

client.load_table_from_dataframe(
    poids_dept,
    "wagon-bootcamp-501621.risques_naturels.projection_couts_par_departement",
    job_config=job_config
).result()
print("✓ projection_couts_par_departement exportée")

         alea  frequence_annuelle_projetee  cout_moyen_par_arrete_eur  \
0  Inondation                         1780                     242237   
1  Secheresse                         2066                     667315   
2  MvtTerrain                          125                      31006   

   cout_projete_annuel_eur  cout_projete_5ans_eur  cout_projete_10ans_eur  
0                431084586             2155422932              4310845864  
1               1378740418             6893702088             13787404177  
2                  3875691               19378457                38756914  
  departement_code          departement_nom  cout_inondation_est  \
0               01                      ain             19700000   
1               02                    aisne             96750000   
2               03                   allier             16000000   
3               04  alpes-de-haute-provence             15600000   
4               05             hautes-alpes             1045000

In [ ]:
df_catnat["lib_risque_jo_clean"] = df_catnat["lib_risque_jo"].str.lower().str.strip()
df_catnat["cat"] = df_catnat["lib_risque_jo_clean"].map(mapping)

In [ ]:

# Restructurer en format long pour un sélecteur unique................

poids_dept_long = poids_dept.melt(
    id_vars=["departement_code", "departement_nom"],
    value_vars=["projection_inondation_annuelle_eur", "projection_secheresse_annuelle_eur", "projection_mvtterrain_annuelle_eur"],
    var_name="alea",
    value_name="cout_projete_annuel_eur"
)

# Nettoyer les noms d'aléa pour qu'ils soient lisibles dans le sélecteur.....................................
poids_dept_long["alea"] = poids_dept_long["alea"].map({
    "projection_inondation_annuelle_eur": "Inondation",
    "projection_secheresse_annuelle_eur": "Secheresse",
    "projection_mvtterrain_annuelle_eur": "MvtTerrain",
})

print(poids_dept_long.head(10))
print("\nDimensions:", poids_dept_long.shape)

# Export vers BigQuery.....................................................................
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
client.load_table_from_dataframe(
    poids_dept_long,
    "wagon-bootcamp-501621.risques_naturels.projection_couts_par_departement_long",
    job_config=job_config
).result()
print("✓ projection_couts_par_departement_long exportée")

  departement_code          departement_nom        alea  \
0               01                      ain  Inondation   
1               02                    aisne  Inondation   
2               03                   allier  Inondation   
3               04  alpes-de-haute-provence  Inondation   
4               05             hautes-alpes  Inondation   
5               06          alpes-maritimes  Inondation   
6               07                  ardeche  Inondation   
7               08                 ardennes  Inondation   
8               09                   ariege  Inondation   
9               10                     aube  Inondation   

   cout_projete_annuel_eur  
0                 443588.0  
1                2178537.0  
2                 360275.0  
3                 351268.0  
4                 235304.0  
5               27774937.0  
6                2121118.0  
7                4976296.0  
8                 637236.0  
9                 878170.0  

Dimensions: (288, 4)
✓ project